## Import

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


## Load data

In [2]:
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
test.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


## Data Quality Check

In [4]:
def missing_report(df, only_missing=True):
    rep = pd.DataFrame({
        "dtype": df.dtypes,
        "n_missing": df.isna().sum(),
        "missing_rate": df.isna().mean().round(4),
        "n_unique": df.nunique(),
    }).reindex(df.columns)          # 元の列順を明示的に固定
    return rep[rep["n_missing"] > 0] if only_missing else rep

In [5]:
print(missing_report(train))

                dtype  n_missing  missing_rate  n_unique
HomePlanet     object        201        0.0231         3
CryoSleep      object        217        0.0250         2
Cabin          object        199        0.0229      6560
Destination    object        182        0.0209         3
Age           float64        179        0.0206        80
VIP            object        203        0.0234         2
RoomService   float64        181        0.0208      1273
FoodCourt     float64        183        0.0211      1507
ShoppingMall  float64        208        0.0239      1115
Spa           float64        183        0.0211      1327
VRDeck        float64        188        0.0216      1306
Name           object        200        0.0230      8473


In [6]:
print(missing_report(test))

                dtype  n_missing  missing_rate  n_unique
HomePlanet     object         87        0.0203         3
CryoSleep      object         93        0.0217         2
Cabin          object        100        0.0234      3265
Destination    object         92        0.0215         3
Age           float64         91        0.0213        79
VIP            object         93        0.0217         2
RoomService   float64         82        0.0192       842
FoodCourt     float64        106        0.0248       902
ShoppingMall  float64         98        0.0229       715
Spa           float64        101        0.0236       833
VRDeck        float64         80        0.0187       796
Name           object         94        0.0220      4176


## EDA

In [7]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

na_mask = train["CryoSleep"].isna()
total = train.loc[na_mask, spend_cols].sum(axis=1, min_count=1)
#df["CryoSleep"].isna().sum()


In [8]:
len(train[na_mask & (total == 0)])

98

In [9]:
len(train[na_mask & (total > 0)])

119

## Preprocess

### RoomServiceが低い or 0の場合はCryoSleppはTrue

In [10]:
def impute_CryoSleep(df):
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df = df.copy()
    na_mask = df["CryoSleep"].isna()
    total = df.loc[na_mask, spend_cols].sum(axis=1, min_count=1)
    
    df.loc[na_mask & (total == 0), "CryoSleep"] = True
    df.loc[na_mask & (total > 0), "CryoSleep"] = False
    return df

In [11]:
train = impute_CryoSleep(train)
test = impute_CryoSleep(test)

### PassengerIDからHomePlanetを推測

In [12]:
def fill_vip_by_spend(df, spend_cols, tol=0.25):
    """VIP=True の支出合計の中央値を基準に、±tol 以内の行の VIP 欠損を True で埋める。"""
    df = df.copy()
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    total = df[spend_cols].sum(axis=1, min_count=1)

    # 基準となる中央値（VIP=True の行のみ、合計が NaN の行は除外）
    med = total[df["VIP"] == True].median()
    low, high = med * (1 - tol), med * (1 + tol)

    na_mask = df["VIP"].isna()
    hit = na_mask & total.between(low, high)
    df.loc[hit, "VIP"] = True

    print(f"median={med:.1f}, range=[{low:.1f}, {high:.1f}], filled={hit.sum()} / {na_mask.sum()}")
    return df

In [13]:
train = fill_vip_by_spend(train, spend_cols)
test = fill_vip_by_spend(test, spend_cols)

median=2767.0, range=[2075.2, 3458.8], filled=16 / 203
median=2713.5, range=[2035.1, 3391.9], filled=6 / 93


## Predict

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score## Predict

In [15]:
from lightgbm import LGBMClassifier

def run_lgbm(n_estimators, max_depth, learning_rate=0.1, num_leaves=31):
    model = LGBMClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,      # 制限なしは -1
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        random_state=0,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, y_pred

In [16]:
y_train = train["Transported"]
features = ["CryoSleep", "VIP", "Age"]

In [17]:
X_train = pd.get_dummies(train[features])
X_test = pd.get_dummies(test[features])

In [18]:
model, y_pred = run_lgbm(100,5)

## Evaluate

In [19]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
scores = cross_val_score(model, X_train, y_train, cv=cv, n_jobs=-1)
print(f"{scores.mean():.4f} ± {scores.std():.4f}  (n={len(scores)})")

0.7403 ± 0.0078  (n=50)


## Output

In [20]:
output = pd.DataFrame({
    "PassengerId": test.PassengerId,
    "Transported": y_pred
})
output.to_csv("submission.csv", index=False)
print("Save csv")

Save csv
